# EDA 11 · Label audit: customers who renew early and are still labelled as churn

**Question.** KKBox defines churn as "no new valid subscription within 30 days after the expiry". A probe after
notebook 10 found manual payers who **pay again before their expiry**, with a new expiry past the label month and
**no cancellation afterwards**, yet 40 % of them are labelled as churn. They make up about a quarter of all manual
churners. Is this label **noise**, which no feature can recover, or a **rule** we do not know, which features of the
renewing payment can learn?

This matters most for the per-cycle setting of notebook 10 (`as_of = expiry − k`, now the priority). There, the early
payment is often visible at `as_of`, and a model that trusts it will predict "stays" for these customers.

**Experiments.**
- **E0 · Is it stable?** The same measurement in every February split (`fit`, `stop`, `tune`) and in the March
  cohort. Rates only; nothing is trained.
- **E1 · Is it learnable?** Inside the anomalous group, predict the label from the renewing payment and the state
  before it. A depth-4 tree (readable rules) and gradient boosting (upper bound) are trained on `fit` and evaluated
  on `stop + tune` and on March.
  Decision rule: AUC ≥ 0.8 means a systematic rule (turn it into features for the per-cycle model); AUC ≈ 0.5–0.6
  means irreducible noise (quantify the ceiling; consider cleaning the training set only).

**Label audit, not feature work.** The analysis uses records after the reference date (the renewal itself) and the
labels of every split, including March. That is legitimate here because nothing is selected for a model from these
results; they only describe what the label means.

In [1]:
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier, export_text

EDA_DIR = next(p for p in (Path.cwd(), Path.cwd() / "eda") if (p / "_common.py").exists())
sys.path.insert(0, str(EDA_DIR))
import _common as C

pd.options.display.float_format = "{:,.4f}".format
pl.Config.set_tbl_rows(40)
FEB_REF, MAR_REF = date(2017, 1, 31), date(2017, 2, 28)
TXN_END = date(2017, 3, 31)          # last transaction date available in either file

## 1. The renewal after the anchor expiry

For every in-scope customer (anchor expiry inside the label month, as in notebook 10):
- **renewal**: the first payment dated after the reference date whose expiry goes past the anchor (records
  collapsed by the cleaning rules of notebook 02; ties broken by expiry, then file order);
- `gap` = renewal date − anchor (≤ 0 means paid on or before the expiry);
- **cancel after**: any cancellation record dated after the reference date.

Groups:

| group | meaning | label expected by the definition |
|---|---|---|
| `early, no cancel` | paid on/before the expiry, never cancelled afterwards | 0 |
| `early + cancel` | paid on/before the expiry, then switched off auto-renew | depends on the new expiry |
| `late 1-30` | paid 1–30 days after the expiry | 0 |
| `late 31+` | paid more than 30 days after the expiry | 1 |
| `none seen` | no renewal by 2017-03-31 | 1 (February); unknown for March, whose window ends 2017-04-30 |

In [2]:
def audit_table(users, ref, max_days):
    C.set_cutoff(ref)
    st = C.subscription_state(C.clean_transactions(users.select("msno", "is_churn")), users)
    st = st.filter(pl.col("days_to_expire").is_between(1, max_days)).select(
        "msno", "is_churn", "split", "state", anchor="last_expire",
        prev_plan="last_plan_days", prev_method="last_payment_method_id", prev_price_type="last_price_type")
    C.set_cutoff(TXN_END)
    t = C.clean_transactions(st.select("msno"))
    C.set_cutoff(FEB_REF)
    post = t.filter(pl.col("td") > pl.lit(ref)).join(st.select("msno", "anchor"), on="msno")
    renew = (post.filter((pl.col("is_cancel") == 0) & (pl.col("ed") > pl.col("anchor")))
             .sort("td", "ed", "_ord")
             .group_by("msno").agg(
                 r_date=pl.col("td").first(), r_expire=pl.col("ed").first(),
                 r_plan=pl.col("payment_plan_days").first(), r_list=pl.col("plan_list_price").first(),
                 r_paid=pl.col("actual_amount_paid").first(), r_price_type=pl.col("price_type").first(),
                 r_method=pl.col("payment_method_id").first(), r_auto=pl.col("is_auto_renew").first(),
                 r_source=pl.col("source").first(), r_same_day_records=pl.len()))
    n_pay = post.filter(pl.col("is_cancel") == 0).group_by("msno").agg(n_pay_after_ref=pl.len())
    canc = post.filter(pl.col("is_cancel") == 1).group_by("msno").agg(first_cancel=pl.col("td").min())
    d = (st.join(renew, on="msno", how="left").join(n_pay, on="msno", how="left").join(canc, on="msno", how="left")
         .with_columns(gap=(pl.col("r_date") - pl.col("anchor")).dt.total_days(),
                       n_pay_after_ref=pl.col("n_pay_after_ref").fill_null(0)))
    return d.with_columns(group=pl.when(pl.col("gap").is_null()).then(pl.lit("none seen"))
        .when((pl.col("gap") <= 0) & pl.col("first_cancel").is_null()).then(pl.lit("early, no cancel"))
        .when(pl.col("gap") <= 0).then(pl.lit("early + cancel"))
        .when(pl.col("gap") <= 30).then(pl.lit("late 1-30"))
        .otherwise(pl.lit("late 31+")))

feb = audit_table(C.load_cohort(), FEB_REF, 28)
mar_users = pl.read_parquet(C.PARQUET_DIR / "train_v2.parquet", columns=["msno", "is_churn"]).with_columns(split=pl.lit("march"))
mar = audit_table(mar_users, MAR_REF, 31)
aud = pl.concat([feb, mar], how="vertical_relaxed")
print(f"in scope: February {feb.height:,} | March {mar.height:,}")

in scope: February 883,631 | March 865,793


## 2. E0 · Is the anomaly stable across splits and months?

In [3]:
GROUPS = ["early, no cancel", "early + cancel", "late 1-30", "late 31+", "none seen"]
SPLITS = ["fit", "stop", "tune", "march"]
tab = (aud.filter(pl.col("state").is_in(["manual", "auto-renew"]))
       .group_by("state", "group", "split").agg(n=pl.len(), churn=pl.col("is_churn").mean()).to_pandas())
rate = tab.pivot_table(index=["state", "group"], columns="split", values="churn")[SPLITS]
size = tab.pivot_table(index=["state", "group"], columns="split", values="n")[SPLITS]
print("churn rate by group and split")
display(rate.reindex([(s, g) for s in ("manual", "auto-renew") for g in GROUPS]))
print("customers by group and split")
size.reindex([(s, g) for s in ("manual", "auto-renew") for g in GROUPS]).astype("Int64")

churn rate by group and split


split                          fit   stop   tune  march
state      group                                       
manual     early, no cancel 0.3985 0.3986 0.3949 0.4494
           early + cancel   0.2937 0.1905 0.2609 0.5763
           late 1-30        0.0036 0.0047 0.0048 0.1468
           late 31+         1.0000 1.0000 1.0000    NaN
           none seen        1.0000 1.0000 1.0000 0.7455
auto-renew early, no cancel 0.0002 0.0002 0.0002 0.0058
           early + cancel   0.1749 0.1716 0.1738 0.4473
           late 1-30        0.0039 0.0044 0.0034 0.0111
           late 31+         0.9868 1.0000 1.0000    NaN
           none seen        1.0000 1.0000 1.0000 0.9322

customers by group and split


split                           fit   stop   tune   march
state      group                                         
manual     early, no cancel   15104   2589   2474   23060
           early + cancel       143     21     23      59
           late 1-30          46573   7819   7714   51953
           late 31+            1593    238    242    <NA>
           none seen          15073   2555   2534   30029
auto-renew early, no cancel  453106  75440  75354  621123
           early + cancel     22435   3754   3769   11812
           late 1-30          93536  15605  15713  107221
           late 31+             151     22     23    <NA>
           none seen           7698   1252   1275   10602

In [4]:
# how much of the churn does the anomaly account for?
share = (aud.filter(pl.col("is_churn") == 1)
         .with_columns(anomalous=(pl.col("group") == "early, no cancel"))
         .group_by("split", "state").agg(churners=pl.len(), anomalous=pl.col("anomalous").sum())
         .with_columns(share=pl.col("anomalous") / pl.col("churners"))
         .filter(pl.col("state").is_in(["manual", "auto-renew"])).sort("state", "split"))
tot = (aud.filter(pl.col("is_churn") == 1).group_by("split")
       .agg(churners=pl.len(), anomalous=(pl.col("group") == "early, no cancel").sum())
       .with_columns(share=pl.col("anomalous") / pl.col("churners")).sort("split"))
display(share.to_pandas())
tot.to_pandas()

,split,state,churners,anomalous,share
0,fit,auto-renew,12224,93,0.0076
1,march,auto-renew,19986,3628,0.1815
2,stop,auto-renew,2003,17,0.0085
3,tune,auto-renew,2023,16,0.0079
4,fit,manual,22894,6019,0.2629
5,march,manual,40415,10364,0.2564
6,stop,manual,3866,1032,0.2669
7,tune,manual,3796,977,0.2574


,split,churners,anomalous,share
0,fit,36903,6201,0.1680
1,march,63123,14186,0.2247
2,stop,6158,1063,0.1726
3,tune,6094,1004,0.1648


**March caveat.** The March label looks 30 days past expiries that end in March, up to 2017-04-30, but transactions
stop on 2017-03-31. So March `late 1-30` and `none seen` are incomplete. The group that matters, `early, no cancel`,
is fully observed: the renewal happens before the expiry. A cancellation after 2017-03-31 could still move a March
customer out of it, which would make the March rate slightly too high.

## 3. E1 · Is the label learnable inside the anomalous group?

In [5]:
grp = aud.filter((pl.col("group") == "early, no cancel") & (pl.col("state") == "manual")).with_columns(
    days_early=(pl.col("anchor") - pl.col("r_date")).dt.total_days(),
    extension=(pl.col("r_expire") - pl.col("anchor")).dt.total_days(),
    span=(pl.col("r_expire") - pl.col("r_date")).dt.total_days(),
    stacked=((pl.col("r_expire") - pl.col("anchor")).dt.total_days() == pl.col("r_plan")).cast(pl.Int8),
    restarted=((pl.col("r_expire") - pl.col("r_date")).dt.total_days() == pl.col("r_plan")).cast(pl.Int8),
    same_method=(pl.col("r_method") == pl.col("prev_method")).cast(pl.Int8),
    same_plan=(pl.col("r_plan") == pl.col("prev_plan")).cast(pl.Int8),
    r_day_of_month=pl.col("r_date").dt.day(),
    expire_month=pl.col("r_expire").dt.month(),
    r_discount=(pl.col("r_list") - pl.col("r_paid")),
)
NUMF = ["days_early", "extension", "span", "stacked", "restarted", "same_method", "same_plan", "r_plan", "r_list",
        "r_paid", "r_discount", "r_auto", "r_source", "r_day_of_month", "n_pay_after_ref", "prev_plan"]
CATF = ["r_method", "r_price_type", "prev_price_type"]
print(grp.group_by("split").agg(n=pl.len(), churn=pl.col("is_churn").mean()).sort("split"))

shape: (4, 3)
┌───────┬───────┬──────────┐
│ split ┆ n     ┆ churn    │
│ ---   ┆ ---   ┆ ---      │
│ str   ┆ u32   ┆ f64      │
╞═══════╪═══════╪══════════╡
│ fit   ┆ 15104 ┆ 0.398504 │
│ march ┆ 23060 ┆ 0.449436 │
│ stop  ┆ 2589  ┆ 0.39861  │
│ tune  ┆ 2474  ┆ 0.394907 │
└───────┴───────┴──────────┘


In [6]:
# univariate: AUC of each numeric candidate (direction-free) on fit, and churn by the main categorical levels
fitg = grp.filter(pl.col("split") == "fit")
uni = []
for c in NUMF:
    x = fitg[c].cast(pl.Float64).fill_null(-1).to_numpy()
    if np.unique(x).size > 1:
        a = roc_auc_score(fitg["is_churn"].to_numpy(), x)
        uni.append({"feature": c, "auc (direction-free)": max(a, 1 - a), "n_unique": int(np.unique(x).size)})
display(pd.DataFrame(uni).sort_values("auc (direction-free)", ascending=False).reset_index(drop=True))
for c in ["r_plan", "stacked", "restarted", "r_source", "r_auto", "same_plan"]:
    display(C.churn_rate_by(fitg.with_columns(pl.col(c).cast(pl.String).fill_null("null")), c)
            .query("n >= 100")[[c, "n", "churn_rate", "ci_low", "ci_high"]])

,feature,auc (direction-free),n_unique
0,span,0.8222,224
1,days_early,0.8095,28
2,restarted,0.7792,2
3,extension,0.7230,62
4,r_paid,0.7079,28
5,r_list,0.7075,28
6,r_plan,0.7064,19
7,r_source,0.7040,2
8,n_pay_after_ref,0.6792,12
9,prev_plan,0.6477,22


,r_plan,n,churn_rate,ci_low,ci_high
1,100,138,0.6812,0.5994,0.7531
3,180,708,0.8305,0.8011,0.8563
4,195,723,0.6943,0.6598,0.7268
7,30,10881,0.2614,0.2532,0.2697
10,395,336,0.9345,0.9029,0.9564
12,410,1036,0.7847,0.7587,0.8087
18,90,814,0.6818,0.6490,0.7129


,stacked,n,churn_rate,ci_low,ci_high
0,0,2163,0.5589,0.5379,0.5797
1,1,12941,0.3717,0.3634,0.3801


,restarted,n,churn_rate,ci_low,ci_high
0,0,9980,0.6011,0.5915,0.6107
1,1,5124,0.0039,0.0025,0.0060


,r_source,n,churn_rate,ci_low,ci_high
0,1,10917,0.2632,0.2550,0.2715
1,2,4187,0.7514,0.7381,0.7642


,r_auto,n,churn_rate,ci_low,ci_high
0,0,14418,0.4170,0.4090,0.4250
1,1,686,0.0102,0.0050,0.0209


,same_plan,n,churn_rate,ci_low,ci_high
0,0,3272,0.6079,0.5910,0.6245
1,1,11832,0.3406,0.3321,0.3492


In [7]:
def matrix(df, levels=None):
    p = df.select(NUMF + CATF).to_pandas()
    levels = levels or {c: sorted(v for v in p[c].dropna().unique()) for c in CATF}
    for c in CATF:
        codes = pd.Categorical(p[c], categories=levels[c]).codes.astype(float)
        codes[codes < 0] = np.nan
        p[c] = codes
    return p.astype(float), levels

Xf, LV = matrix(fitg)
yf = fitg["is_churn"].to_numpy()
parts = {"Feb holdout (stop+tune)": grp.filter(pl.col("split").is_in(["stop", "tune"])), "March": grp.filter(pl.col("split") == "march")}

tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, random_state=C.SEED).fit(Xf.fillna(-1), yf)
gbm = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, categorical_features=[len(NUMF) + i for i in range(len(CATF))],
                                     random_state=C.SEED).fit(Xf, yf)
rows = []
for name, part in parts.items():
    Xp, _ = matrix(part, LV)
    yp = part["is_churn"].to_numpy()
    rows.append({"evaluated on": name, "n": len(yp), "churn rate": yp.mean(),
                 "tree depth 4 AUC": roc_auc_score(yp, tree.predict_proba(Xp.fillna(-1))[:, 1]),
                 "gradient boosting AUC": roc_auc_score(yp, gbm.predict_proba(Xp)[:, 1])})
e1 = pd.DataFrame(rows)
e1

/tmp/ipykernel_4/3128095782.py:5: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  codes = pd.Categorical(p[c], categories=levels[c]).codes.astype(float)


,evaluated on,n,churn rate,tree depth 4 AUC,gradient boosting AUC
0,Feb holdout (stop+tune),5063,0.3968,0.9306,0.9439
1,March,23060,0.4494,0.7447,0.6754


In [8]:
print(export_text(tree, feature_names=NUMF + CATF, show_weights=True, decimals=1))

|--- days_early <= 0.5
|   |--- span <= 29.5
|   |   |--- weights: [203.0, 6.0] class: 0
|   |--- span >  29.5
|   |   |--- r_plan <= 362.5
|   |   |   |--- prev_plan <= 110.0
|   |   |   |   |--- weights: [4449.0, 0.0] class: 0
|   |   |   |--- prev_plan >  110.0
|   |   |   |   |--- weights: [515.0, 1.0] class: 0
|   |   |--- r_plan >  362.5
|   |   |   |--- weights: [273.0, 2.0] class: 0
|--- days_early >  0.5
|   |--- span <= 65.5
|   |   |--- n_pay_after_ref <= 1.5
|   |   |   |--- r_day_of_month <= 10.5
|   |   |   |   |--- weights: [1.0, 203.0] class: 1
|   |   |   |--- r_day_of_month >  10.5
|   |   |   |   |--- weights: [0.0, 427.0] class: 1
|   |   |--- n_pay_after_ref >  1.5
|   |   |   |--- r_auto <= 0.5
|   |   |   |   |--- weights: [3127.0, 2255.0] class: 0
|   |   |   |--- r_auto >  0.5
|   |   |   |   |--- weights: [477.0, 5.0] class: 0
|   |--- span >  65.5
|   |   |--- n_pay_after_ref <= 1.5
|   |   |   |--- prev_plan <= 45.0
|   |   |   |   |--- weights: [1.0, 422.0]

### 3.1 E1b · An explicit rule for the label

The tree says three things:
- paying **on** the expiry day means staying;
- paying **before** it, with no further payment, means churn;
- long plans paid early mean churn.

That points to a definition in which only a payment **dated on or after the expiry** counts as a renewal. The cell
below tests explicit rules against the label on the whole February in-scope cohort, using transactions up to
2017-03-31. The 30-day window ends on 2017-03-30 at the latest, so it is fully observed. March cannot be tested: its
window runs to 2017-04-30.

In [9]:
C.set_cutoff(TXN_END)
tf = C.clean_transactions(feb.select("msno")).join(feb.select("msno", "anchor"), on="msno").filter(pl.col("td") > pl.lit(FEB_REF))
C.set_cutoff(FEB_REF)
pay = pl.col("is_cancel") == 0
window = (pl.col("td") >= pl.col("anchor")) & (pl.col("td") <= pl.col("anchor") + pl.duration(days=30))
RULES = {
    "R1 any payment extending past the expiry": pay & (pl.col("ed") > pl.col("anchor")),
    "R2 a payment dated in [expiry, expiry + 30]": pay & window,
    "R3 any record (payment or cancel) dated in [expiry, expiry + 30]": window,
}
rows, confusion = [], {}
for name, flag in RULES.items():
    hit = tf.filter(flag).select("msno").unique().with_columns(hit=pl.lit(True))
    d = feb.join(hit, on="msno", how="left").with_columns(pred=pl.col("hit").is_null().cast(pl.Int8))
    rows.append({"rule (stays if ...)": name, "agreement with is_churn": (d["pred"] == d["is_churn"]).mean(),
                 "label 1, rule 0": int(((d["pred"] == 0) & (d["is_churn"] == 1)).sum()),
                 "label 0, rule 1": int(((d["pred"] == 1) & (d["is_churn"] == 0)).sum())})
    confusion[name] = d
display(pd.DataFrame(rows))
d = confusion["R2 a payment dated in [expiry, expiry + 30]"]
(d.group_by("state", "pred", "is_churn").len().to_pandas()
 .pivot_table(index=["state", "pred"], columns="is_churn", values="len").fillna(0).astype(int))

,rule (stays if ...),agreement with is_churn,"label 1, rule 0","label 0, rule 1"
0,R1 any payment extending past the expiry,0.9812,16601,0
1,"R2 a payment dated in [expiry, expiry + 30]",0.9937,5409,176
2,R3 any record (payment or cancel) dated in [ex...,0.9883,10157,160


is_churn                    0      1
state            pred               
auto-renew       0     752709   4263
                 1        174  11987
cancel           0       1177     19
                 1          0   1946
cancel + payment 0       6277     10
                 1          0    374
manual           0      74137   1117
                 1          2  29439

## 5. Why do early payers churn 6 % in February and 42 % in March? (follow-up to notebook 10, section 6.1)

Notebook 10 grouped the customers who paid ahead before `as_of` by how far the payment extends the subscription
(`ahead_extension`). The 26–30-day bucket churned 6 % in February and 42 % in March. The cell below rebuilds the
group at k = 1 (as_of = anchor − 1) and splits it by the **exact** extension and by the type of payment.

In [10]:
def early_payers(audit, ref):
    C.set_cutoff(TXN_END)
    t = C.clean_transactions(audit.select("msno")).join(audit.select("msno", "anchor"), on="msno")
    C.set_cutoff(FEB_REF)
    ah = (t.filter((pl.col("is_cancel") == 0) & (pl.col("ed") > pl.col("anchor")) & (pl.col("td") <= pl.col("anchor") - pl.duration(days=1)))
          .sort("td", "ed", "_ord").group_by("msno")
          .agg(a_auto=pl.col("is_auto_renew").first(), a_max=pl.col("ed").max()))
    return (audit.select("msno", "is_churn", "state", "anchor").join(ah, on="msno")
            .with_columns(ext=(pl.col("a_max") - pl.col("anchor")).dt.total_days(),
                          payment=pl.when(pl.col("a_auto") == 1).then(pl.lit("auto-renew charge")).otherwise(pl.lit("manual payment"))))

ep = pl.concat([early_payers(feb, FEB_REF).with_columns(cohort=pl.lit("February")),
                early_payers(mar, MAR_REF).with_columns(cohort=pl.lit("March"))])
bucket = ep.filter(pl.col("ext").is_between(26, 30))
display(bucket.group_by("cohort").agg(n=pl.len(), churn=pl.col("is_churn").mean()).sort("cohort").to_pandas())
(ep.filter(pl.col("ext").is_between(26, 33))
   .group_by("cohort", "ext", "payment").agg(n=pl.len(), churn=pl.col("is_churn").mean())
   .filter(pl.col("n") >= 100).sort("ext", "payment", "cohort").to_pandas()
   .pivot_table(index=["ext", "payment"], columns="cohort", values=["n", "churn"]))

,cohort,n,churn
0,February,84014,0.0631
1,March,11349,0.4223


churn                  n            
cohort                February  March    February       March
ext payment                                                  
26  auto-renew charge   0.0270 0.0520    333.0000    173.0000
27  auto-renew charge   0.0492 0.0142    630.0000    211.0000
28  auto-renew charge   0.0219 0.0225 75,095.0000    267.0000
29  auto-renew charge      NaN 0.0117         NaN    256.0000
30  auto-renew charge      NaN 0.2280         NaN    715.0000
    manual payment      0.4633 0.4743  7,789.0000  9,692.0000
31  auto-renew charge      NaN 0.0544         NaN 80,542.0000

**A composition effect of month lengths, not a label mechanic.** At the exact extension the two months behave the
same:
- auto-renew charges taken early churn about 2 % in both;
- manual 30-day plans paid early churn about 46 % in both.

What changes is who falls in the bucket. An early monthly auto-renew charge extends the subscription to the same day
next month: **+28 days in February**, inside the 26–30 bucket, which dilutes it to 6 %, and **+31 days in March**,
outside it, which leaves only the risky manual payers (42 %).

**Consequence for the per-cycle model.** `days_to_expire` at `as_of` mixes k with month length once a customer has
renewed early. The same normal auto-renewer reads k + 28 in February and k + 31 in March. A model trained on February
can then place March auto-renewers on the side of its split learned for the risky manual payers (k + 30). This is
the same calendar drift notebook 08 fixed with `expiry_month`, and it grows at small k, where early payments are
common (14 % of customers at k = 1 in March). It is a candidate explanation for the March F1 plateau. A
calendar-robust encoding has to be tested in notebook 10.

## 6. Conclusion

**The anomaly is not noise. It is the label definition.** The KKBox label counts as a renewal only a payment **dated
between the expiry and 30 days after it**. A manual payment made *before* the expiry does not count, even when it
extends the subscription. Rule R2 reproduces `is_churn` for **99.4 %** of the February in-scope cohort. The rule
"any payment that extends past the expiry", which is what a business would call a renewal, matches only 98.1 %.

- **E0 · stable.** Manual early renewers without a cancellation churn 39.5–39.9 % in `fit`, `stop` and `tune`, and
  45 % in March. They are 26 % of the manual churners in every split, and 17 % of all in-scope churners (22 % in
  March, inflated by cancellations after 2017-03-31 that we cannot see).
- **E1 · learnable.** Inside that group the label is predicted with AUC 0.93 (depth-4 tree) and 0.94 (gradient
  boosting) on the February holdout. The tree reads like the rule:
  - paid **on** the expiry day → stays (5,440 vs 9);
  - paid **before** it with no further payment → churn (3,012 vs 2);
  - paid before it with a later payment → mixed, depending on whether that payment lands inside the window.
- **March is weaker** (AUC 0.74 / 0.68) because the audit features stop on 2017-03-31, before the March window
  closes. This does not affect the per-cycle model, whose `as_of` is always before the expiry.
- **What it means.** About 1 in 6 churners in the label are customers who **prepaid**. They are not lost customers.
  A model must learn KKBox's rule to score well on the official label. A retention campaign should not target them.

**Recommendation for the per-cycle model (priority).**
1. Keep the official label as the target and in every reported metric. March only has that label.
2. Add the features that let the model learn the rule. They are visible at `as_of` whenever the early payment
   happened before it:
   - `paid_ahead`: a payment in the current cycle, dated before `as_of`, that extends past the expiry;
   - `days_early` of that payment;
   - the plan length of that payment;
   - the number of payments in the current cycle.

   Expected effect: the plateau of the March F1 in notebook 10 should partly lift.
3. Do not drop these customers from training. The pattern is learnable, and dropping them biases the official metric.
4. Report, next to it, a February-only diagnostic with the "business" label (R1). The March window cannot be
   observed, so that label cannot be built for March.

In [11]:
C.write_decisions("11", "Label audit", [
    {"finding": "is_churn = no payment dated in [expiry, expiry + 30] (99.4 % agreement); a payment made before the expiry does not count as renewal even when it extends the subscription",
     "decision": "Keep the official label as target and metric; document the definition in the w1/w2 reports", "kind": "no action"},
    {"finding": "Manual early renewers without cancellation churn 40 % (45 % March); stable across fit/stop/tune; 26 % of manual churners, ~17 % of all in-scope churners",
     "decision": "Treat them as a systematic part of the label, not noise; do not drop them from training", "kind": "no action"},
    {"finding": "Inside that group the label is learnable (AUC 0.93 tree / 0.94 GBM on the Feb holdout) from payment timing and plan length",
     "decision": "Per-cycle model: add paid_ahead, days_early, plan days of the early payment and payments in the current cycle (all dated <= as_of)", "kind": "new feature"},
    {"finding": "Early payers extending 26-30 days churn 6 % (Feb) vs 42 % (Mar) only because an early monthly auto-renew charge extends +28 days in February and +31 in March; at the exact extension both months match (2 % auto, 46 % manual)",
     "decision": "No label mechanic to model; but days_to_expire after an early payment carries month length (k+28 vs k+31): test a calendar-robust encoding in the per-cycle model", "kind": "open"},
    {"finding": "A business label (any payment extending past the expiry) differs for ~1 in 6 churners and needs data to expiry + 30 (unavailable for March)",
     "decision": "Report a February-only diagnostic with the business label next to the official metrics", "kind": "open"},
])

,finding,decision,kind
0,"is_churn = no payment dated in [expiry, expiry...",Keep the official label as target and metric; ...,no action
1,Manual early renewers without cancellation chu...,"Treat them as a systematic part of the label, ...",no action
2,Inside that group the label is learnable (AUC ...,"Per-cycle model: add paid_ahead, days_early, p...",new feature
3,Early payers extending 26-30 days churn 6 % (F...,No label mechanic to model; but days_to_expire...,open
4,A business label (any payment extending past t...,Report a February-only diagnostic with the bus...,open
